In [ ]:

# 安装必要的库
!pip install lightgbm pandas scikit-learn


D:\software\Python\PyCharm\PyCharm 20250201\PyCharm 2025.2.0.1\plugins\python-ce\helpers\pycharm_display\datalore\display\supported_data_type.py:6: UserWarning: The NumPy module was reloaded (imported a second time). This can in some cases result in small but subtle issues and is discouraged.
  import numpy


In [ ]:

# 导入必要的库
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 加载数据
train_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/reservation_cancellation/train.csv'
test_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/reservation_cancellation/test.csv'

train_df = pd.read_csv(train_data_path)
test_df = pd.read_csv(test_data_path)

# 查看数据的基本信息
print("训练集信息：")
print(train_df.head())
print("\n测试集信息：")
print(test_df.head())


训练集信息：
      id  no_of_adults  ...  no_of_special_requests  booking_status
0  15559             2  ...                       2               0
1  32783             2  ...                       1               0
2  11797             3  ...                       0               1
3  39750             2  ...                       1               1
4  28711             2  ...                       0               1

[5 rows x 19 columns]

测试集信息：
      id  no_of_adults  ...  no_of_special_requests  booking_status
0   8768             2  ...                       1               0
1  38340             2  ...                       0               1
2   7104             2  ...                       0               0
3  36898             2  ...                       3               0
4   9747             2  ...                       1               0

[5 rows x 19 columns]


In [ ]:


# 数据预处理
# 检查是否有缺失值
print("训练集缺失值：")
print(train_df.isnull().sum())
print("\n测试集缺失值：")
print(test_df.isnull().sum())

# 编码分类变量
label_encoders = {}
for column in train_df.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    train_df[column] = le.fit_transform(train_df[column])
    test_df[column] = le.transform(test_df[column])
    label_encoders[column] = le

# 分割特征和标签
X_train = train_df.drop(columns=['id', 'booking_status'])
y_train = train_df['booking_status']
X_test = test_df.drop(columns=['id'])



训练集缺失值：
id                                      0
no_of_adults                            0
no_of_children                          0
no_of_weekend_nights                    0
no_of_week_nights                       0
type_of_meal_plan                       0
required_car_parking_space              0
room_type_reserved                      0
lead_time                               0
arrival_year                            0
arrival_month                           0
arrival_date                            0
market_segment_type                     0
repeated_guest                          0
no_of_previous_cancellations            0
no_of_previous_bookings_not_canceled    0
avg_price_per_room                      0
no_of_special_requests                  0
booking_status                          0
dtype: int64

测试集缺失值：
id                                      0
no_of_adults                            0
no_of_children                          0
no_of_weekend_nights                    0
no_o

In [ ]:



# 训练LightGBM模型
train_dataset = lgb.Dataset(X_train, label=y_train)
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9
}
num_round = 100

bst = lgb.train(params, train_dataset, num_round)

# 预测
y_pred = bst.predict(X_test)

# 评估模型
y_true = test_df['booking_status']
auc_roc = roc_auc_score(y_true, y_pred)
print(f"模型在测试集上的AUC-ROC: {auc_roc:.4f}")

# 保存预测结果
test_df['predicted_booking_status'] = y_pred
output_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\predictions/reservation_cancellation_predictions.csv'
test_df.to_csv(output_path, index=False)



Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

  1181         num_iteration=num_iteration,
   1182         predict_type=predict_type,
   1183     )
   1184 elif isinstance(data, np.ndarray):
-> 1185     preds, nrow = self.__pred_for_np2d(
   1186         mat=data,
   1187         start_iteration=start_iteration,
   1188         num_iteration=num_iteration,
   1189         predict_type=predict_type,
   1190     )
   1191 elif _is_pyarrow_table(data):
   1192     preds, nrow = self.__pred_for_pyarrow_table(
   1193         table=data,
   1194         start_iteration=start_iteration,
   1195         num_iteration=num_iteration,
   1196         predict_type=predict_type,
   1197     )

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\lightgbm\basic.py:1344, in _InnerPredictor.__pred_for_np2d(self, mat, start_iteration, num_iteration, 

In [ ]:


# 重新检查特征列
train_features = set(X_train.columns)
test_features = set(X_test.columns)

# 找出差异
missing_in_test = train_features - test_features
missing_in_train = test_features - train_features

print(f"特征缺失在训练集：{missing_in_train}")
print(f"特征缺失在测试集：{missing_in_test}")

# 确保Test集包含训练集中所有特征
for col in missing_in_test:
    X_test[col] = 0  # 或者其他合适的默认值

# 重新训练LightGBM模型
train_dataset = lgb.Dataset(X_train, label=y_train)
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9
}
num_round = 100

bst = lgb.train(params, train_dataset, num_round)

# 预测
y_pred = bst.predict(X_test)

# 评估模型
y_true = test_df['booking_status']
auc_roc = roc_auc_score(y_true, y_pred)
print(f"模型在测试集上的AUC-ROC: {auc_roc:.4f}")

# 保存预测结果
test_df['predicted_booking_status'] = y_pred
output_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\predictions/reservation_cancellation_predictions.csv'
test_df.to_csv(output_path, index=False)




Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

  1181         num_iteration=num_iteration,
   1182         predict_type=predict_type,
   1183     )
   1184 elif isinstance(data, np.ndarray):
-> 1185     preds, nrow = self.__pred_for_np2d(
   1186         mat=data,
   1187         start_iteration=start_iteration,
   1188         num_iteration=num_iteration,
   1189         predict_type=predict_type,
   1190     )
   1191 elif _is_pyarrow_table(data):
   1192     preds, nrow = self.__pred_for_pyarrow_table(
   1193         table=data,
   1194         start_iteration=start_iteration,
   1195         num_iteration=num_iteration,
   1196         predict_type=predict_type,
   1197     )

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\lightgbm\basic.py:1344, in _InnerPredictor.__pred_for_np2d(self, mat, start_iteration, num_iteration, 

In [ ]:



# 重新检查特征列
train_features = sorted(X_train.columns)
test_features = sorted(X_test.columns)

print("训练集特征列：")
print(train_features)

print("\n测试集特征列：")
print(test_features)

# 检查特征列是否有差异
if train_features != test_features:
    print("\n训练集和测试集的特征列有差异，将进行调整。")
    # 找出差值
    missing_in_test = [col for col in train_features if col not in test_features]
    missing_in_train = [col for col in test_features if col not in train_features]

    # 调整测试集，使其具有训练集的所有特征列
    for col in missing_in_test:
        X_test[col] = 0  # 或者其他合适的默认值

    # 确保Training集不包含测试集特有的列
    X_train = X_train[[col for col in X_train if col in test_features]]

    # 重新训练LightGBM模型
    train_dataset = lgb.Dataset(X_train, label=y_train)
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.05,
        'feature_fraction': 0.9
    }
    num_round = 100

    bst = lgb.train(params, train_dataset, num_round)

    # 预测
    y_pred = bst.predict(X_test)

    # 评估模型
    auc_roc = roc_auc_score(y_true, y_pred)
    print(f"模型在测试集上的AUC-ROC: {auc_roc:.4f}")

    # 保存预测结果
    test_df['predicted_booking_status'] = y_pred
    output_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\predictions/reservation_cancellation_predictions.csv'
    test_df.to_csv(output_path, index=False)

else:
    print("\n训练集和测试集的特征列一致，无需调整。")





Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

  1181         num_iteration=num_iteration,
   1182         predict_type=predict_type,
   1183     )
   1184 elif isinstance(data, np.ndarray):
-> 1185     preds, nrow = self.__pred_for_np2d(
   1186         mat=data,
   1187         start_iteration=start_iteration,
   1188         num_iteration=num_iteration,
   1189         predict_type=predict_type,
   1190     )
   1191 elif _is_pyarrow_table(data):
   1192     preds, nrow = self.__pred_for_pyarrow_table(
   1193         table=data,
   1194         start_iteration=start_iteration,
   1195         num_iteration=num_iteration,
   1196         predict_type=predict_type,
   1197     )

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\lightgbm\basic.py:1344, in _InnerPredictor.__pred_for_np2d(self, mat, start_iteration, num_iteration, 

In [ ]:




# 重新检查特征列
train_features = sorted(X_train.columns)
test_features = sorted(X_test.columns)

print("训练集特征列：")
print(train_features)

print("\n测试集特征列：")
print(test_features)

# 检查特征列是否有差异
if train_features != test_features:
    print("\n训练集和测试集的特征列有差异，将进行调整。")
    # 找出差值
    missing_in_test = [col for col in train_features if col not in test_features]
    missing_in_train = [col for col in test_features if col not in train_features]

    # 调整测试集，使其具有训练集的所有特征列
    for col in missing_in_test:
        X_test[col] = 0  # 或者其他合适的默认值

    # 确保Training集不包含测试集特有的列
    X_train = X_train[[col for col in X_train if col in test_features]]

    # 重新训练LightGBM模型
    train_dataset = lgb.Dataset(X_train, label=y_train)
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.05,
        'feature_fraction': 0.9
    }
    num_round = 100

    bst = lgb.train(params, train_dataset, num_round)

    # 预测
    y_pred = bst.predict(X_test)

    # 评估模型
    auc_roc = roc_auc_score(y_true, y_pred)
    print(f"模型在测试集上的AUC-ROC: {auc_roc:.4f}")

    # 保存预测结果
    test_df['predicted_booking_status'] = y_pred
    output_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\predictions/reservation_cancellation_predictions.csv'
    test_df.to_csv(output_path, index=False)

else:
    print("\n训练集和测试集的特征列一致，无需调整。")

# 校验特征数量是否一致
print(f"\n训练集特征数量: {X_train.shape[1]}")
print(f"测试集特征数量: {X_test.shape[1]}")




Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

  1181         num_iteration=num_iteration,
   1182         predict_type=predict_type,
   1183     )
   1184 elif isinstance(data, np.ndarray):
-> 1185     preds, nrow = self.__pred_for_np2d(
   1186         mat=data,
   1187         start_iteration=start_iteration,
   1188         num_iteration=num_iteration,
   1189         predict_type=predict_type,
   1190     )
   1191 elif _is_pyarrow_table(data):
   1192     preds, nrow = self.__pred_for_pyarrow_table(
   1193         table=data,
   1194         start_iteration=start_iteration,
   1195         num_iteration=num_iteration,
   1196         predict_type=predict_type,
   1197     )

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\lightgbm\basic.py:1344, in _InnerPredictor.__pred_for_np2d(self, mat, start_iteration, num_iteration, 

In [ ]:




# 重新检查特征列
train_features = sorted(X_train.columns)
test_features = sorted(X_test.columns)

print("训练集特征列：")
print(train_features)

print("\n测试集特征列：")
print(test_features)

# 检查特征列是否有差异
if train_features != test_features:
    print("\n训练集和测试集的特征列有差异。")
    # 找出差值
    missing_in_test = [col for col in train_features if col not in test_features]
    missing_in_train = [col for col in test_features if col not in train_features]

    # 调整测试集，使其具有训练集的所有特征列
    for col in missing_in_test:
        X_test[col] = 0  # 或者其他合适的默认值

    # 确保Training集不包含测试集特有的列
    X_train = X_train[[col for col in X_train if col in test_features]]

    # 重新训练LightGBM模型
    train_dataset = lgb.Dataset(X_train, label=y_train)
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.05,
        'feature_fraction': 0.9
    }
    num_round = 100

    bst = lgb.train(params, train_dataset, num_round)

    # 预测
    y_pred = bst.predict(X_test)

    # 评估模型
    auc_roc = roc_auc_score(y_true, y_pred)
    print(f"模型在测试集上的AUC-ROC: {auc_roc:.4f}")

    # 保存预测结果
    test_df['predicted_booking_status'] = y_pred
    output_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\predictions/reservation_cancellation_predictions.csv'
    test_df.to_csv(output_path, index=False)

else:
    print("\n训练集和测试集的特征列一致，无需调整。")

# 校验特征数量是否一致
print(f"\n训练集特征数量: {X_train.shape[1]}")
print(f"测试集特征数量: {X_test.shape[1]}")

# 打印详细的前几行数据以进一步检查
print("\n训练集前几行数据：")
print(train_df.head())

print("\n测试集前几行数据：")
print(test_df.head())




Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

  1181         num_iteration=num_iteration,
   1182         predict_type=predict_type,
   1183     )
   1184 elif isinstance(data, np.ndarray):
-> 1185     preds, nrow = self.__pred_for_np2d(
   1186         mat=data,
   1187         start_iteration=start_iteration,
   1188         num_iteration=num_iteration,
   1189         predict_type=predict_type,
   1190     )
   1191 elif _is_pyarrow_table(data):
   1192     preds, nrow = self.__pred_for_pyarrow_table(
   1193         table=data,
   1194         start_iteration=start_iteration,
   1195         num_iteration=num_iteration,
   1196         predict_type=predict_type,
   1197     )

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\lightgbm\basic.py:1344, in _InnerPredictor.__pred_for_np2d(self, mat, start_iteration, num_iteration, 

In [ ]:




# 重新加载数据集并确保一致性
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\processed/reservation_cancellation_processed.csv'
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\processed/reservation_cancellation_test_processed.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# 分离特征和标签
X_train, y_train = train_df.drop('booking_status', axis=1), train_df['booking_status']
X_test = test_df.drop('booking_status', axis=1)

# 重新检查特征列
train_features = sorted(X_train.columns)
test_features = sorted(X_test.columns)

print("训练集特征列（排序后）：")
print(train_features)

print("\n测试集特征列（排序后）：")
print(test_features)

# 检查特征列是否有差异
if train_features != test_features:
    print("\n训练集和测试集的特征列有差异，将进行调整。")
    # 找出差值
    missing_in_test = [col for col in train_features if col not in test_features]
    missing_in_train = [col for col in test_features if col not in train_features]

    # 调整测试集，使其具有训练集的所有特征列
    for col in missing_in_test:
        X_test[col] = 0  # 或者其他合适的默认值

    # 确保Training集不包含测试集特有的列
    X_train = X_train[[col for col in X_train if col in test_features]]

    # 重新训练LightGBM模型
    train_dataset = lgb.Dataset(X_train, label=y_train)
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.05,
        'feature_fraction': 0.9
    }
    num_round = 100

    bst = lgb.train(params, train_dataset, num_round)

    # 预测
    y_pred = bst.predict(X_test)

    # 评估模型
    y_true = test_df.drop('booking_status', axis=1)
    auc_roc = roc_auc_score(y_true, y_pred)
    print(f"模型在测试集上的AUC-ROC: {auc_roc:.4f}")

    # 保存预测结果
    test_df['predicted_booking_status'] = y_pred
    output_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\predictions/reservation_cancellation_predictions.csv'
    test_df.to_csv(output_path, index=False)

else:
    print("\n训练集和测试集的特征列一致，无需调整。")

# 校验特征数量是否一致
print(f"\n训练集特征数量: {X_train.shape[1]}")
print(f"测试集特征数量: {X_test.shape[1]}")

# 再次打印详细的前几行数据以进一步检查
print("\n训练集前几行数据：")
print(train_df.head())

print("\n测试集前几行数据：")
print(test_df.head())






Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

ing, doublequote, escapechar, comment, encoding, encoding_errors, dialect, on_bad_lines, delim_whitespace, low_memory, memory_map, float_precision, storage_options, dtype_backend)
   1013 kwds_defaults = _refine_defaults_read(
   1014     dialect,
   1015     delimiter,
   (...)
   1022     dtype_backend=dtype_backend,
   1023 )
   1024 kwds.update(kwds_defaults)
-> 1026 return _read(filepath_or_buffer, kwds)

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\pandas\io\parsers\readers.py:620, in _read(filepath_or_buffer, kwds)
    617 _validate_names(kwds.get("names", None))
    619 # Create the parser.
--> 620 parser = TextFileReader(filepath_or_buffer, **kwds)
    622 if chunksize or iterator:
    623     return parser

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter

In [ ]:

import os

# Define file paths
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\processed\reservation_cancellation_processed.csv'
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\processed\reservation_cancellation_test_processed.csv'

# Check if files exist
train_file_exists = os.path.exists(train_path)
test_file_exists = os.path.exists(test_path)

print(f"训练集文件存在: {train_file_exists}")
print(f"测试集文件存在: {test_file_exists}")


训练集文件存在: False
测试集文件存在: False
